In [5]:
import pandas as pd

trades = pd.read_csv(r"C:\Users\S . Mariya Nisha\Downloads\historical_data.csv")
sentiment = pd.read_csv(r"C:\Users\S . Mariya Nisha\Downloads\fear_greed_index.csv")

In [6]:
trades.shape
sentiment.shape

(2644, 4)

In [7]:
trades.isnull().sum()
sentiment.isnull().sum()

timestamp         0
value             0
classification    0
date              0
dtype: int64

In [8]:
trades.duplicated().sum()
sentiment.duplicated().sum()

0

In [15]:
trades.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [16]:
trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], dayfirst=True)

In [17]:
trades['date'] = trades['Timestamp IST'].dt.date

In [23]:
merged = pd.merge(trades, sentiment, on='date', how='inner')

In [24]:
merged.groupby(['Account', 'date'])['Closed PnL'].sum()

Series([], Name: Closed PnL, dtype: float64)

In [25]:
merged.groupby('date').size()

Series([], dtype: int64)

In [26]:
merged['Side'].value_counts()

Series([], Name: count, dtype: int64)

In [27]:
merged['Size USD'].mean()

nan

In [29]:
merged.groupby('classification')['Closed PnL'].mean()

Series([], Name: Closed PnL, dtype: float64)

In [30]:
merged.groupby(['classification', 'date']).size()

Series([], dtype: int64)

In [32]:
merged.groupby(['classification', 'Side']).size()

Series([], dtype: int64)

In [35]:
merged.groupby('classification').size()

Series([], dtype: int64)

In [36]:
merged.groupby('classification')['Size USD'].mean()

Series([], Name: Size USD, dtype: float64)

In [37]:
merged.groupby(['classification', 'Side']).size()

Series([], dtype: int64)

In [38]:
merged.groupby('classification')['Closed PnL'].mean()

Series([], Name: Closed PnL, dtype: float64)

In [40]:
trade_count = merged.groupby('Account').size()

In [42]:
frequent = trade_count[trade_count > trade_count.median()]
infrequent = trade_count[trade_count <= trade_count.median()]

In [43]:
pnl = merged.groupby('Account')['Closed PnL'].sum()

In [44]:
winners = pnl[pnl > 0]
losers = pnl[pnl <= 0]

In [45]:
size = merged.groupby('Account')['Size USD'].mean()

In [46]:
large = size[size > size.median()]
small = size[size <= size.median()]

In [47]:
merged['profit_flag'] = merged['Closed PnL'].apply(lambda x: 1 if x > 0 else 0)

In [48]:
result = merged.groupby('classification')['profit_flag'].mean()
print(result)

Series([], Name: profit_flag, dtype: float64)


In [49]:
def simple_strategy(row):
    if row['classification'] == 'Fear':
        return "Reduce trading / smaller size"
    else:
        return "Trade actively but manage risk"

merged['strategy'] = merged.apply(simple_strategy, axis=1)

In [50]:
from sklearn.cluster import KMeans